In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SCHEMA = "workspace.lakehouse_demo"

def save_table(df, name):
    (
        df.write.format("delta")
        .mode("overwrite")
        .saveAsTable(f"{SCHEMA}.{name}")
    )

def split_quality(df, rules):
    # rules : liste de (motif, condition d'invalidité)
    checked = df.withColumn(
        "_reject_reason",
        F.concat_ws(
            "; ",
            *[
                F.when(F.coalesce(condition, F.lit(False)), F.lit(reason))
                for reason, condition in rules
            ]
        )
    )

    valid = checked.filter(F.col("_reject_reason") == "").drop("_reject_reason")
    rejected = checked.filter(F.col("_reject_reason") != "")
    return valid, rejected

def mapping_expr(column, mapping):
    entries = []
    for key, value in mapping.items():
        entries.extend([F.lit(key), F.lit(value)])
    return F.element_at(F.create_map(*entries), column)

def invalid_number(column):
    c = F.col(column)
    return (
        c.isNull()
        | F.isnan(c)
        | (F.abs(c) == F.lit(float("inf")))
        | (c < 0)
    )

In [0]:
customers = spark.table(f"{SCHEMA}.bronze_customers")

# Empreinte du contenu brut pour départager les ex-aequo.
content_columns = sorted(
    c for c in customers.columns if not c.startswith("_")
)

customers = customers.withColumn(
    "_row_hash",
    F.sha2(
        F.to_json(
            F.struct(*[F.col(f"`{c}`") for c in content_columns]),
            options={"ignoreNullFields": "false"}
        ),
        256
    )
)

window = Window.partitionBy("customer_id").orderBy(
    F.col("_ingested_at").desc(),
    F.col("_row_hash").asc()
)

ranked = customers.withColumn("_rn", F.row_number().over(window))

save_table(
    ranked.filter(F.col("_rn") > 1).drop("_rn", "_row_hash"),
    "audit_customer_duplicates"
)

customers = (
    ranked.filter(F.col("_rn") == 1)
    .drop("_rn", "_row_hash")
)

for name in ["first_name", "last_name", "city", "state"]:
    customers = customers.withColumn(name, F.initcap(F.trim(F.col(name))))

for name in ["email", "country"]:
    customers = customers.withColumn(name, F.lower(F.trim(F.col(name))))

customers = (
    customers
    .withColumn("dob", F.expr("try_cast(dob AS DATE)"))
    .withColumn("signup_date", F.expr("try_cast(signup_date AS DATE)"))
)

valid_customers, rejected_customers = split_quality(customers, [
    ("customer_id absent", F.col("customer_id").isNull()),
    ("email absent", F.col("email").isNull()),
])

save_table(valid_customers, "silver_customers")
save_table(rejected_customers, "reject_customers")

In [0]:
categories = [
    "electronics", "sports", "beauty", "automotive",
    "home", "kitchen", "toys", "clothing"
]

products = (
    spark.table(f"{SCHEMA}.bronze_products")
    .withColumn("_raw_category", F.col("category"))
    .withColumn("_raw_price", F.col("price"))
    .withColumn(
        "_category_clean",
        F.regexp_replace(
            F.regexp_replace(
                F.lower(F.trim(F.col("category"))),
                r"[-_]+$", ""
            ),
            "3", "e"
        )
    )
    .withColumn("price", F.expr("try_cast(trim(price) AS DOUBLE)"))
)

# Même priorité que la boucle du code local.
category_result = F.lit(None).cast("string")

for name in reversed(categories):
    condition = F.expr(
        f"""
        _category_clean <> ''
        AND (
            _category_clean = '{name}'
            OR startswith('{name}', _category_clean)
            OR startswith(_category_clean, '{name}')
        )
        """
    )
    category_result = F.when(condition, F.lit(name)).otherwise(category_result)

products = products.withColumn("category", category_result).drop("_category_clean")

valid_products, rejected_products = split_quality(products, [
    ("product_id absent", F.col("product_id").isNull()),
    ("prix invalide", invalid_number("price")),
    ("catégorie non reconnue", F.col("category").isNull()),
])

# Pas de déduplication arbitraire : une clé répétée multiplierait les jointures.
duplicate_products = (
    valid_products.groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_products.limit(1).count() > 0:
    display(duplicate_products.limit(20))
    raise ValueError(
        "Des product_id sont répétés. Définir une règle avant de continuer."
    )

save_table(valid_products, "silver_products")
save_table(rejected_products, "reject_products")

In [0]:
STATUS_MAP = {
    "success": "success", "suc": "success",
    "failed": "failed", "fail": "failed",
    "refunded": "refunded", "ref": "refunded",
}

PAYMENT_MAP = {
    "wallet": "wallet",
    "cash": "cash",
    "upi": "upi",
    "card": "card", "crad": "card", "crd": "card",
}

QUANTITY_MAP = {
    word: str(i)
    for i, word in enumerate(
        ["one", "two", "three", "four", "five",
         "six", "seven", "eight", "nine", "ten"],
        start=1
    )
}

orders = spark.table(f"{SCHEMA}.bronze_orders")

for name in ["quantity", "status", "payment_method", "order_amount", "order_date"]:
    orders = orders.withColumn(f"_raw_{name}", F.col(name))

orders = (
    orders
    .withColumn(
        "status",
        mapping_expr(F.lower(F.trim(F.col("status"))), STATUS_MAP)
    )
    .withColumn(
        "payment_method",
        mapping_expr(
            F.regexp_replace(
                F.lower(F.trim(F.col("payment_method"))), "[^a-z]", ""
            ),
            PAYMENT_MAP
        )
    )
    .withColumn(
        "_quantity_text",
        F.coalesce(
            mapping_expr(
                F.lower(F.trim(F.col("quantity"))), QUANTITY_MAP
            ),
            F.trim(F.col("quantity"))
        )
    )
    .withColumn(
        "quantity",
        F.expr("try_cast(try_cast(_quantity_text AS FLOAT) AS INT)")
    )
    .withColumn(
        "order_amount",
        F.expr("try_cast(trim(order_amount) AS DOUBLE)")
    )
    .withColumn(
        "order_date",
        F.when(
            F.trim(F.col("_raw_order_date")).rlike(r"^\d{4}-\d{2}-\d{2}$"),
            F.expr("try_cast(trim(_raw_order_date) AS DATE)")
        ).otherwise(F.lit(None).cast("date"))
    )
    .drop("_quantity_text")
)

valid_orders, rejected_orders = split_quality(orders, [
    ("quantité invalide", F.col("quantity").isNull() | (F.col("quantity") <= 0)),
    ("statut non reconnu", F.col("status").isNull()),
    ("paiement non reconnu", F.col("payment_method").isNull()),
    ("montant invalide", invalid_number("order_amount")),
    ("date invalide selon la règle du dataset", F.col("order_date").isNull()),
])

save_table(valid_orders, "silver_orders")
save_table(rejected_orders, "reject_orders")

In [0]:
orders = spark.table(f"{SCHEMA}.silver_orders").alias("o")
customers = spark.table(f"{SCHEMA}.silver_customers").alias("c")
products = spark.table(f"{SCHEMA}.silver_products").alias("p")

joined = (
    orders
    .join(customers, F.col("o.customer_id") == F.col("c.customer_id"), "left")
    .join(products, F.col("o.product_id") == F.col("p.product_id"), "left")
    .select(
        "o.*",
        F.col("c.city").alias("customer_city"),
        F.col("c.country").alias("customer_country"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("catalog_price"),
        F.col("c.customer_id").isNotNull().alias("_customer_found"),
        F.col("p.product_id").isNotNull().alias("_product_found"),
    )
)

enriched, orphans = split_quality(joined, [
    ("client absent de Silver", ~F.col("_customer_found")),
    ("produit absent de Silver", ~F.col("_product_found")),
])

save_table(
    enriched.drop("_customer_found", "_product_found"),
    "silver_orders_enriched"
)

save_table(
    orphans.drop("_customer_found", "_product_found"),
    "reject_order_references"
)

In [0]:
table_names = [
    "silver_customers",
    "audit_customer_duplicates",
    "reject_customers",
    "silver_products",
    "reject_products",
    "silver_orders",
    "reject_orders",
    "silver_orders_enriched",
    "reject_order_references",
]

counts = [
    (name, spark.table(f"{SCHEMA}.{name}").count())
    for name in table_names
]

display(spark.createDataFrame(counts, ["table", "nombre_lignes"]))

table,nombre_lignes
silver_customers,47200
audit_customer_duplicates,1800
reject_customers,1000
silver_products,405
reject_products,95
silver_orders,150137
reject_orders,149863
silver_orders_enriched,118909
reject_order_references,31228
